# CEMC and Li Percolation Example for Li1.2Mn0.4Ti0.4O2

This notebook reproduces the CEMC-to-percolation workflow for one representative composition, Li1.2Mn0.4Ti0.4O2. By default it generates 5 CEMC structures at each of 100 K, 1000 K, 10000 K, and 100000 K, then calculates the percolating Li fraction for each structure and plots a Fig. 2b-style summary.

For manuscript-scale statistics, change `N_STRUCTURES` from 5 to 100.

## Dependencies

Install the required packages before running:

```bash
pip install git+https://github.com/Liaojh123/SROS.git
pip install git+https://github.com/CederGroupHub/smol.git
pip install git+https://github.com/atomisticnet/dribble.git
pip install pymatgen numpy pandas matplotlib
```

`SROS` generates the random starting structures and runs the CEMC workflow, `smol` is required by the cluster-expansion Monte Carlo code, and `dribble` performs the Li percolation calculation.

In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from SROS import CEMC, generate_random_any_composition
from dribble.io import Input
from dribble.lattice import Lattice
from dribble.percolator import Percolator

# Run this notebook from the examples/ directory. If you run it elsewhere,
# replace these paths with absolute paths, e.g. /home/ljh/.../model.mson.
ce_model_path = "../models/Li-Mn-Ti-O_clustre_expansion_model.mson"
output_dir = "cemc_percolation_Li1p2Mn0p4Ti0p4O2"

li_content = 1.2
mn_content = 0.4
temperatures = [100, 1000, 10000, 100000]
N_STRUCTURES = 5
MC_STEPS = 120000
SC_MATRIX = np.array([[5, 0, 0], [0, 6, 0], [0, 0, 8]])
BASE_SEED = 202502

Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"CE model: {ce_model_path}")
print(f"Output directory: {output_dir}")

## Generate CEMC Structures

Each structure starts from a random Li-Mn-Ti-O DRX configuration and is relaxed by CEMC at the target temperature using the provided cluster-expansion model.

In [ ]:
def make_initial_structure(seed):
    random.seed(seed)
    np.random.seed(seed)
    supercell = generate_random_any_composition.make_supercell(scaling_matrix=SC_MATRIX.tolist())
    return generate_random_any_composition.modify_structure(li_content, mn_content, supercell)


def generate_cemc_structures(overwrite=False):
    structure_files = []
    for temperature in temperatures:
        temp_dir = Path(output_dir) / f"{temperature}K"
        temp_dir.mkdir(parents=True, exist_ok=True)
        for i in range(N_STRUCTURES):
            out_file = temp_dir / f"CEMC_{i:03d}.vasp"
            if overwrite or not out_file.exists():
                seed = BASE_SEED + temperature + i
                initial_structure = make_initial_structure(seed)
                cemc_structure = CEMC.run_cemc(
                    li_content,
                    mn_content,
                    initial_structure,
                    ce_model_path,
                    sc_matrix=SC_MATRIX,
                    temperature=temperature,
                    mc_step=MC_STEPS,
                )
                cemc_structure.to(str(out_file), fmt="poscar")
            structure_files.append((temperature, i, out_file))
    return structure_files


structure_files = generate_cemc_structures(overwrite=False)
print(f"Generated or found {len(structure_files)} CEMC structures.")

## Calculate Percolating Li Fraction

The percolation setup follows the manuscript workflow: Li sites are the percolating cation sublattice, Mn/Ti occupy the transition-metal cation sublattice, oxygen is ignored, and Li-Li connectivity uses the `MinCommonNNNeighborsBR` rule with two common nearest neighbors.

In [ ]:
def write_dribble_input(json_file, structure_file):
    content = {
        "structure": str(Path(structure_file).resolve()),
        "formula_units": 1,
        "sublattices": {
            "cations": {
                "description": "Cation sites",
                "sites": {"species": ["Li"]},
                "initial_occupancy": {"Li": 1.0},
            },
            "cation2": {
                "description": "Transition-metal cation sites",
                "sites": {"species": ["Mn", "Ti"]},
                "initial_occupancy": {"TM": 1.0},
            },
            "oxygen": {
                "description": "Oxygen sites",
                "sites": {"species": ["O"]},
                "ignore": True,
            },
        },
        "bonds": [
            {
                "sublattices": ["cations", "cations"],
                "bond_rules": [["MinCommonNNNeighborsBR", {"num_neighbors": 2}]],
            }
        ],
        "percolating_species": ["Li"],
        "flip_sequence": [["TM", "Li"]],
    }
    Path(json_file).write_text(json.dumps(content, indent=4), encoding="utf-8")


def calculate_percolating_fraction(structure_file):
    input_json = Path(output_dir) / "input-bond-rule.json"
    write_dribble_input(input_json, structure_file)

    inp = Input.from_file(str(input_json))
    lattice = Lattice.from_input_object(inp, supercell=(1, 1, 1))
    percolator = Percolator.from_input_object(inp, lattice, verbose=False)
    n_li = percolator.num_occupied
    n_span = percolator.check_spanning(
        verbose=False,
        save_clusters=False,
        static_sites=inp.static_sites,
    )
    return n_span / n_li if n_li else np.nan


rows = []
for temperature, i, structure_file in structure_files:
    fraction = calculate_percolating_fraction(structure_file)
    rows.append({
        "temperature_K": temperature,
        "structure_index": i,
        "structure": str(structure_file),
        "percolating_li_fraction": fraction,
        "percolating_li_percent": 100 * fraction,
    })

results = pd.DataFrame(rows)
results

## Notes

The generated `CEMC_*.vasp` structures are written under `cemc_percolation_Li1p2Mn0p4Ti0p4O2/`. This folder is ignored by git because it is generated output. Increase `N_STRUCTURES` to 100 to reproduce the manuscript-scale statistics.
